In [45]:
import io
import os
import requests
import pandas as pd
from google.cloud import storage

In [46]:

# services = ['fhv','green','yellow']
init_url = 'https://github.com/DataTalksClub/nyc-tlc-data/releases/download/'

In [47]:
i = 1
year = "2020"
service = "fhv"

In [48]:
# sets the month part of the file_name string
month = '0'+str(i+1)
month = month[-2:]

# csv file_name
file_name = f"{service}_tripdata_{year}-{month}.csv.gz"

# download it using requests via a pandas df
request_url = f"{init_url}{service}/{file_name}"
r = requests.get(request_url)
open(file_name, 'wb').write(r.content)
print(f"Local: {file_name}")

Local: fhv_tripdata_2020-02.csv.gz


In [49]:
!ls

Define_schema.ipynb         docker-compose.yaml
Dockerfile                  environment.yaml
README.md                   fhv_tripdata_2020-02.csv.gz
assets                      taxi_rides_ny
dbt                         terraform
dbt.md                      web_to_gcs.py


In [52]:
df = pd.read_csv(file_name, compression='gzip',low_memory=False)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa0 in position 148352: invalid start byte

In [7]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge
0,2.0,2019-07-01 00:51:04,2019-07-01 00:51:33,1.0,0.00,1.0,N,193,193,1.0,2.5,0.5,0.5,1.14,0.00,0.3,4.94,0.0
1,2.0,2019-07-01 00:46:04,2019-07-01 01:05:46,1.0,4.16,1.0,N,234,25,2.0,16.5,0.5,0.5,0.00,0.00,0.3,20.30,2.5
2,1.0,2019-07-01 00:25:09,2019-07-01 01:00:56,1.0,18.80,2.0,N,132,42,1.0,52.0,0.0,0.5,11.75,6.12,0.3,70.67,0.0
3,2.0,2019-07-01 00:33:32,2019-07-01 01:15:27,1.0,18.46,2.0,N,132,142,1.0,52.0,0.0,0.5,11.06,0.00,0.3,66.36,2.5
4,1.0,2019-07-01 00:00:55,2019-07-01 00:13:05,0.0,1.70,1.0,N,107,114,1.0,9.5,3.0,0.5,2.00,0.00,0.3,15.30,2.5


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6310419 entries, 0 to 6310418
Data columns (total 18 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   VendorID               float64
 1   tpep_pickup_datetime   object 
 2   tpep_dropoff_datetime  object 
 3   passenger_count        float64
 4   trip_distance          float64
 5   RatecodeID             float64
 6   store_and_fwd_flag     object 
 7   PULocationID           int64  
 8   DOLocationID           int64  
 9   payment_type           float64
 10  fare_amount            float64
 11  extra                  float64
 12  mta_tax                float64
 13  tip_amount             float64
 14  tolls_amount           float64
 15  improvement_surcharge  float64
 16  total_amount           float64
 17  congestion_surcharge   float64
dtypes: float64(13), int64(2), object(3)
memory usage: 866.6+ MB


In [9]:
# cast time data to timestamp
time_cols = df.filter(regex='datetime').columns
print(list(time_cols))
for column in time_cols:
    df[column] = pd.to_datetime(df[column])

['tpep_pickup_datetime', 'tpep_dropoff_datetime']


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6310419 entries, 0 to 6310418
Data columns (total 18 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               float64       
 1   tpep_pickup_datetime   datetime64[ns]
 2   tpep_dropoff_datetime  datetime64[ns]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int64         
 8   DOLocationID           int64         
 9   payment_type           float64       
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
dtypes: datetime64[ns](2), 

In [11]:
# also cast IDs and types to integer
id_cols = df.filter(regex='ID|type|count').columns
print(list(id_cols))
for column in id_cols:
    print(column)
    df[column] = df[column].astype('Int64')

['VendorID', 'passenger_count', 'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type']
VendorID
passenger_count
RatecodeID
PULocationID
DOLocationID
payment_type


In [12]:
df.VendorID

0             2
1             2
2             1
3             2
4             1
           ... 
6310414    <NA>
6310415    <NA>
6310416    <NA>
6310417    <NA>
6310418    <NA>
Name: VendorID, Length: 6310419, dtype: Int64

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6310419 entries, 0 to 6310418
Data columns (total 18 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               Int64         
 1   tpep_pickup_datetime   datetime64[ns]
 2   tpep_dropoff_datetime  datetime64[ns]
 3   passenger_count        Int64         
 4   trip_distance          float64       
 5   RatecodeID             Int64         
 6   store_and_fwd_flag     object        
 7   PULocationID           Int64         
 8   DOLocationID           Int64         
 9   payment_type           Int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
dtypes: Int64(6), datetime6

In [19]:
df.store_and_fwd_flag.describe()

count     6276460
unique          2
top             N
freq      6231262
Name: store_and_fwd_flag, dtype: object

In [34]:
def fix_types(df):
    # cast time data to timestamp
    for column in df.filter(regex='datetime64[us]').columns:
        df[column] = pd.to_datetime(df[column])
    # also cast IDs and types to integer
    for column in df.filter(regex='ID|type|count').columns:
        df[column] = df[column].astype('Int64') 
    for column in df.filter(regex='flag').columns:
        df[column] = df[column].astype('string') 
    return df

In [39]:
# read it back into a parquet file
df = pd.read_csv(file_name, compression='gzip')


/var/folders/7d/89k5r2594h55__fqkm8m658rxn168q/T/ipykernel_15069/4261694753.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_name, compression='gzip')


In [38]:
df = fix_types(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6310419 entries, 0 to 6310418
Data columns (total 18 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   VendorID               Int64  
 1   tpep_pickup_datetime   object 
 2   tpep_dropoff_datetime  object 
 3   passenger_count        Int64  
 4   trip_distance          float64
 5   RatecodeID             Int64  
 6   store_and_fwd_flag     string 
 7   PULocationID           Int64  
 8   DOLocationID           Int64  
 9   payment_type           Int64  
 10  fare_amount            float64
 11  extra                  float64
 12  mta_tax                float64
 13  tip_amount             float64
 14  tolls_amount           float64
 15  improvement_surcharge  float64
 16  total_amount           float64
 17  congestion_surcharge   float64
dtypes: Int64(6), float64(9), object(2), string(1)
memory usage: 902.7+ MB


In [ ]:
file_name = file_name.replace('.csv.gz', '.parquet')
df.to_parquet(file_name, engine='pyarrow',  coerce_timestamps='us')
print(f"Parquet: {file_name}")

/var/folders/7d/89k5r2594h55__fqkm8m658rxn168q/T/ipykernel_15069/594068888.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_name, compression='gzip')


Parquet: yellow_tripdata_2019-07.parquet


In [31]:
!ls

Define_schema.ipynb             environment.yaml
Dockerfile                      fhv_tripdata_2020-02.csv.gz
README.md                       taxi_rides_ny
assets                          terraform
dbt                             web_to_gcs.py
dbt.md                          yellow_tripdata_2019-07.csv.gz
docker-compose.yaml             yellow_tripdata_2019-07.parquet


In [32]:
# check parquet schema
pq = pd.read_parquet(file_name)

In [33]:
pq.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6310419 entries, 0 to 6310418
Data columns (total 18 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               Int64         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        Int64         
 4   trip_distance          float64       
 5   RatecodeID             Int64         
 6   store_and_fwd_flag     string        
 7   PULocationID           Int64         
 8   DOLocationID           Int64         
 9   payment_type           Int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
dtypes: Int64(6), datetime6

In [27]:
pq.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge
0,2,2019-07-01 00:51:04,2019-07-01 00:51:33,1,0.00,1,N,193,193,1,2.5,0.5,0.5,1.14,0.00,0.3,4.94,0.0
1,2,2019-07-01 00:46:04,2019-07-01 01:05:46,1,4.16,1,N,234,25,2,16.5,0.5,0.5,0.00,0.00,0.3,20.30,2.5
2,1,2019-07-01 00:25:09,2019-07-01 01:00:56,1,18.80,2,N,132,42,1,52.0,0.0,0.5,11.75,6.12,0.3,70.67,0.0
3,2,2019-07-01 00:33:32,2019-07-01 01:15:27,1,18.46,2,N,132,142,1,52.0,0.0,0.5,11.06,0.00,0.3,66.36,2.5
4,1,2019-07-01 00:00:55,2019-07-01 00:13:05,0,1.70,1,N,107,114,1,9.5,3.0,0.5,2.00,0.00,0.3,15.30,2.5


In [68]:
?os.remove

Signature: os.remove(path, *, dir_fd=None)
Docstring:
Remove a file (same as unlink()).

If dir_fd is not None, it should be a file descriptor open to a directory,
  and path should be relative; path will then be relative to that directory.
dir_fd may not be implemented on your platform.
  If it is unavailable, using it will raise a NotImplementedError.
Type:      builtin_function_or_method

In [ ]:
o